# 02 — Types of AI Agents (Code Examples)
Implementing each agent type to understand architectural differences.

**Prerequisites:** `pip install openai python-dotenv rich`

In [ ]:
import os, json
from dotenv import load_dotenv
from openai import OpenAI
from rich import print as rprint
from rich.table import Table

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
print('Setup complete')

## Type 1: Simple Reflex Agent — Condition → Action rules, no memory

In [ ]:
class SimpleReflexAgent:
    """Maps inputs to fixed actions via rules. No history, no planning."""
    def __init__(self):
        self.rules = [
            (lambda x: 'hello' in x.lower(), "Hello! How can I help?"),
            (lambda x: 'bitcoin' in x.lower(), "Check CoinGecko for real-time prices."),
            (lambda x: 'weather' in x.lower(), "Check a weather app for current weather."),
        ]
        self.default = "I don't understand. Type 'help' for options."

    def act(self, user_input: str) -> str:
        for condition, action in self.rules:
            if condition(user_input):
                return action
        return self.default

agent = SimpleReflexAgent()
for q in ["Hello there!", "What's the Bitcoin price?", "Explain consciousness"]:
    print(f"Q: {q}\nA: {agent.act(q)}\n")

print("LIMITATION: Same input always gives same output — zero context.")

## Type 2: Model-Based Agent — Maintains conversation history as state

In [ ]:
class ModelBasedAgent:
    """Uses conversation history (internal state) to give context-aware answers."""
    def __init__(self):
        self.history = []  # Internal state

    def act(self, user_input: str) -> str:
        self.history.append({"role": "user", "content": user_input})
        messages = [{"role": "system", "content": "You are a helpful assistant."}] + self.history
        response = client.chat.completions.create(model="gpt-4o-mini", messages=messages)
        answer = response.choices[0].message.content
        self.history.append({"role": "assistant", "content": answer})
        return answer

agent = ModelBasedAgent()
convos = [
    "My name is Alice and I love machine learning.",
    "What is my name?",
    "What topic did I say I love?",
]
for msg in convos:
    print(f"User: {msg}")
    print(f"Agent: {agent.act(msg)}\n")

print(f"State size: {len(agent.history)} messages in history")

## Type 3: Goal-Based Agent — Plans before acting

In [ ]:
class GoalBasedAgent:
    """Creates an explicit plan before executing any actions."""
    def __init__(self, tools: list[str]):
        self.tools = tools

    def create_plan(self, goal: str) -> list[str]:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": f"Available tools: {self.tools}. Output a numbered list of 3-5 concrete steps to achieve the goal. Be brief."},
                {"role": "user", "content": f"Goal: {goal}"}
            ]
        )
        text = response.choices[0].message.content
        return [l.strip() for l in text.split('\n') if l.strip() and l.strip()[0].isdigit()]

    def pursue_goal(self, goal: str):
        print(f"Goal: {goal}\n")
        plan = self.create_plan(goal)
        print("Plan created:")
        for step in plan:
            print(f"  {step}")
        print(f"\nExecuting {len(plan)} steps... (mocked)")
        for step in plan:
            print(f"  Done: {step[:60]}")

agent = GoalBasedAgent(["web_search", "read_article", "write_file", "send_email"])
agent.pursue_goal("Research the top 3 AI agent frameworks and write a comparison summary")

## Type 4: Utility-Based Agent — Optimizes tradeoffs to select best model

In [ ]:
from dataclasses import dataclass

@dataclass
class LLMOption:
    name: str
    cost: float          # per 1k tokens
    quality: dict        # by task type
    latency_ms: int

MODELS = [
    LLMOption("gpt-4o",           0.0025, {"reasoning": 0.95, "coding": 0.93, "qa": 0.80}, 1200),
    LLMOption("gpt-4o-mini",      0.0001, {"reasoning": 0.75, "coding": 0.78, "qa": 0.85}, 400),
    LLMOption("claude-3-haiku",   0.0001, {"reasoning": 0.72, "coding": 0.74, "qa": 0.88}, 350),
    LLMOption("claude-3-5-sonnet",0.0030, {"reasoning": 0.96, "coding": 0.95, "qa": 0.82}, 1400),
]

def utility(model, task, wq=0.5, wc=0.3, ws=0.2):
    quality = model.quality.get(task, 0.5)
    cost_score = 1 - model.cost / max(m.cost for m in MODELS)
    speed_score = 1 - model.latency_ms / max(m.latency_ms for m in MODELS)
    return wq * quality + wc * cost_score + ws * speed_score

def select_model(task: str, priority: str = "balanced"):
    weights = {
        "quality_first": (0.7, 0.1, 0.2),
        "cost_first":    (0.2, 0.6, 0.2),
        "balanced":      (0.5, 0.3, 0.2),
    }[priority]
    
    ranked = sorted(MODELS, key=lambda m: utility(m, task, *weights), reverse=True)
    print(f"\nTask: '{task}' | Priority: {priority}")
    for m in ranked:
        score = utility(m, task, *weights)
        print(f"  {m.name:22}  score={score:.3f}")
    print(f"  → SELECTED: {ranked[0].name}")

select_model("reasoning", "quality_first")
select_model("qa", "cost_first")
select_model("coding", "balanced")

## Type 5: Learning Agent — Reflexion: attempt → critique → reflect → improve

In [ ]:
class LearningAgent:
    """Implements Reflexion: verbal self-critique stored as long-term memory."""
    
    def __init__(self):
        self.reflections = []

    def attempt(self, task: str) -> str:
        context = ""
        if self.reflections:
            context = "\nPast lessons:\n" + "\n".join(f"- {r}" for r in self.reflections)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Complete this task well. Apply past lessons." + context},
                {"role": "user",   "content": task}
            ]
        )
        return response.choices[0].message.content

    def evaluate_and_reflect(self, task: str, response: str) -> tuple[float, str]:
        result = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content":
                f"Task: {task}\nResponse: {response}\n\n"
                "Rate 0.0-1.0 and state ONE specific improvement in 1 sentence.\n"
                "Format exactly:\nSCORE: 0.X\nLESSON: [one improvement sentence]"
            }]
        ).choices[0].message.content
        
        score, lesson = 0.5, result
        for line in result.split('\n'):
            if line.startswith('SCORE:'):
                try: score = float(line.split(':')[1].strip())
                except: pass
            if line.startswith('LESSON:'):
                lesson = line.replace('LESSON:', '').strip()
        return score, lesson

    def run(self, task: str, threshold=0.85, max_attempts=3):
        print(f"Task: {task}")
        for i in range(1, max_attempts + 1):
            print(f"\n--- Attempt {i} ---")
            resp = self.attempt(task)
            print(f"Response: {resp[:150]}")
            score, lesson = self.evaluate_and_reflect(task, resp)
            print(f"Score: {score:.2f} | Lesson: {lesson[:80]}")
            if score >= threshold:
                print(f"Quality threshold reached!")
                return resp
            self.reflections.append(lesson)
        print(f"Max attempts reached. Reflections learned: {len(self.reflections)}")
        return resp

agent = LearningAgent()
agent.run(
    task="Write exactly 3 bullet points explaining neural networks. Each bullet under 15 words.",
    threshold=0.85
)

## Summary: Agent Types Comparison

In [ ]:
table = Table(title="Agent Types Comparison")
table.add_column("Type", style="bold cyan")
table.add_column("Memory")
table.add_column("Plans")
table.add_column("Learns")
table.add_column("Best For")

for row in [
    ("Simple Reflex",  "❌", "❌", "❌", "FAQ bots, keyword routing"),
    ("Model-Based",    "✅", "❌", "❌", "Multi-turn chatbots"),
    ("Goal-Based",     "✅", "✅", "❌", "Research & automation"),
    ("Utility-Based", "✅", "✅", "❌", "Model routing, optimization"),
    ("Learning",       "✅", "✅", "✅", "Self-improving systems"),
    ("Hierarchical",   "✅", "✅", "✅", "Complex multi-agent"),
]:
    table.add_row(*row)

rprint(table)